In [1]:
path = "../data_base/bbc_text_cls_adapted.csv"

In [2]:
import pandas as pd
import numpy as np
import random
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv(filepath_or_buffer = path)

In [4]:
labels = data["labels"].unique()

In [5]:
X = data["text"]
y = data["labels"]

In [6]:
if y.dtype == object or str:
    flag = True
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [7]:
columns = data.iloc[:,2:]
columns = columns.apply(le.transform)
columns = columns.apply(lambda linha: list(set(linha)), axis=1)

In [8]:
list_cc = []
list_dc = []

for index,_ in enumerate(columns):
    if len(_) == 1:
        list_cc.append(data.iloc[index])
        # list_index_cc.append(index)
    else:
        list_dc.append(data.iloc[index])
        # list_index_cd(_)

df_cc = pd.DataFrame(list_cc)
df_dc = pd.DataFrame(list_dc)

In [9]:
if df_cc.empty == False:
    y_cc = df_cc['labels']
    X_cc = df_cc['text']
    y_dc = df_dc['labels']
    X_dc = df_dc['text']
else:
    y_dc = df_dc['labels']
    X_dc = df_dc['text']

In [10]:
if df_cc.empty == True:
    X_train, X_test, y_train, y_test = train_test_split(X_dc, y_dc, test_size = 0.33, random_state = 1, stratify = y)
else:
    X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(X_cc, y_cc, test_size = 0.35, random_state = 1, stratify = y)
    X_train_dc, X_test_dc, y_train_dc, y_test_dc = train_test_split(X_dc, y_dc, test_size = 0.65, random_state = 1, stratify = y)
    
    X_train = pd.concat([X_train_cc,X_train_dc], ignore_index=True)
    X_test = pd.concat([X_test_cc,X_test_dc], ignore_index=True)
    y_train = pd.concat([y_train_cc,y_train_dc], ignore_index=True)
    y_test = pd.concat([y_test_cc,y_test_dc], ignore_index=True)

In [11]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [12]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [13]:
columns = columns.apply(le.transform)

In [14]:
columns = columns.apply(lambda linha: list(set(linha)), axis=1)

In [15]:
list_cor = np.array(columns)

In [16]:
individual_lenght = (len(data.columns)-2)

In [17]:
individual = np.array([])

for i in range(individual_lenght):
    individual = np.append(individual,random.random())

In [18]:
naive_bayes_mult = MultinomialNB()

In [19]:
def removal_function(individual,n_remove):
    # individual = list(individual)
    indexes = []
    length = len(individual)
    num = length-n_remove
    index = random.randint(0,num)
    indexes.append(index)
    indexes.append(index+1)
    indexes.append(index+2)

    return(indexes)

In [20]:
def insertion_function(indexes,individual): 
    for index in indexes:
        individual[index] = random.random()
        
    return(individual)

In [21]:
def determinator(tiny_list,individual):
    choice = random.choices(tiny_list, weights=individual, k=1)[0]
    return(choice)

In [22]:
def fitness_function(individual,X_train_transformed,list_cor):
    choices = np.array([])
    choices = np.apply_along_axis(lambda x: determinator(x, individual), axis=1, arr=list_cor)

    naive_bayes_mult.fit(X_train_transformed,choices)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)
    list_before_sum = [max(i) for i in df_]  
    fitness = sum(list_before_sum)
    
    return(fitness)

In [23]:
def implementation_function(individual,n_remove,X_train_transformed,list_cor,k):
    best_fitness = float('-inf')
    best_individual = []
    fitness = 0
    
    for _ in range(k):
        indexes = removal_function(individual,n_remove)
        individual = insertion_function(indexes,individual)
        fitness = fitness_function(individual,X_train_transformed,list_cor)
        # print(fitness)
        # print(best_fitness)
        
        if fitness > best_fitness:
            best_individual = individual
            best_fitness = fitness
        else:
            if math.exp(fitness - best_fitness) > random.random():
                best_individual = individual
                best_fitness = fitness

    return(best_fitness,best_individual)

In [24]:
best_fitness,best_individual = implementation_function(individual,3,X_train_transformed,list_cor,1000)

AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
print(best_fitness)
print(best_individual)

-997.3890831345293
[0.648435   0.88221702 0.80948993 0.81335669 0.40603976 0.11804765
 0.04126506 0.65933378 0.05397999 0.79456639 0.550419   0.33711808
 0.76980367 0.58042683 0.5521094  0.39590674 0.75553893 0.64262618
 0.65684011 0.96932052]


In [ ]:
list_cor_ = list_cor.copy()

df_ = pd.DataFrame()
cont = 0
for index,values in enumerate(list_cor_):
    if len(values) > 1:
        list_cor_[index] = best_individual[cont]
        cont+=1

list_cor_ = [item[0] if isinstance(item, list) else item for item in list_cor_]

In [ ]:
if flag == True:
    list_cor_ = le.inverse_transform(list_cor_)
    y_train = le.inverse_transform(y_train)

In [ ]:
confusion_matrix(y_train, list_cor_)